# 30 - Reto 1: Insight Engine (Backend Analitico Gobernado)

## Objetivo del notebook
Construir un motor de insights transparente, auditable y defendible para detectar hallazgos candidatos por zona, **sin causalidad falsa** y respetando el contrato semantico de NB20.

## Relacion con NB20
Este notebook consume reglas de:
- `config/metrics.yaml`
- `config/business_rules.yaml`
- `config/question_types.yaml`
- `reports/reto1/semantic_layer_report.json`

Las reglas de NB20 son vinculantes en este notebook:
- `ZONE_KEY = COUNTRY|CITY|ZONE`
- peer group primario `(COUNTRY, ZONE_TYPE, ZONE_PRIORITIZATION)`
- `lead_penetration` suspendida
- `turbo_adoption` fuera de peer benchmark
- `restaurants_markdowns_gmv` es `lower_is_better`
- direcciones y thresholds aun provisionales

## Outputs esperados
- `reports/reto1/insight_candidates.parquet`
- `reports/reto1/insight_candidates.csv`
- `reports/reto1/insight_engine_report.md`
- `reports/reto1/insight_engine_report.json`
- `reports/reto1/insight_samples.md`

## 0) Setup y contexto metodologico

### Que se hace
Se cargan librerias, rutas y utilidades base del engine.

### Por que se hace
Sin un setup reproducible no hay trazabilidad ni auditabilidad.

### Criterio
- El notebook debe poder ejecutarse de punta a punta.
- No se usan modelos black-box.
- Las decisiones quedan expresadas en funciones y reglas explicitas.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import uuid
import warnings

import numpy as np
import pandas as pd
import yaml
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'config').exists() and (p / 'data' / 'processed').exists():
            return p
    raise FileNotFoundError('No se pudo inferir el root del proyecto.')

ROOT = find_project_root()
CONFIG_DIR = ROOT / 'config'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REPORTS_DIR = ROOT / 'reports' / 'reto1'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('CONFIG_DIR:', CONFIG_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('REPORTS_DIR:', REPORTS_DIR)

ROOT: /home/thechieft/projects/IAeng
CONFIG_DIR: /home/thechieft/projects/IAeng/config
PROCESSED_DIR: /home/thechieft/projects/IAeng/data/processed
REPORTS_DIR: /home/thechieft/projects/IAeng/reports/reto1


## 1) Carga y validacion de inputs gobernados

### Que se hace
Se cargan contratos semanticos, reglas de negocio y datasets procesados.

### Por que se hace
El insight engine no puede operar sin gobernanza semantica. Primero valida contrato, luego detecta.

### Criterio
- Si una metrica esta suspendida, se excluye de detectores no permitidos.
- Si faltan campos de gobernanza (`validation_status`, `direction_confidence`), se detiene.
- Si hay checks FAIL, se explicitan antes de seguir.

In [3]:
with open(CONFIG_DIR / 'metrics.yaml', 'r', encoding='utf-8') as f:
    metrics_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'business_rules.yaml', 'r', encoding='utf-8') as f:
    business_rules = yaml.safe_load(f)
with open(CONFIG_DIR / 'question_types.yaml', 'r', encoding='utf-8') as f:
    question_types_cfg = yaml.safe_load(f)

semantic_report_path = REPORTS_DIR / 'semantic_layer_report.json'
if semantic_report_path.exists():
    with open(semantic_report_path, 'r', encoding='utf-8') as f:
        semantic_report = json.load(f)
else:
    semantic_report = {'semantic_checks': {}}

metrics_long = pd.read_parquet(PROCESSED_DIR / 'metrics_long.parquet')
orders_long = pd.read_parquet(PROCESSED_DIR / 'orders_long.parquet')
zone_master = pd.read_parquet(PROCESSED_DIR / 'zone_master.parquet')

catalog = pd.DataFrame(metrics_cfg['metrics']).copy()
required_metric_fields = [
    'id', 'display_name', 'desired_direction', 'direction_confidence', 'validation_status',
    'low_coverage_peer_groups', 'source_table'
 ]
missing_fields = [c for c in required_metric_fields if c not in catalog.columns]
if missing_fields:
    raise ValueError(f'Faltan campos en metrics.yaml: {missing_fields}')

metrics = metrics_long.merge(
    catalog,
    left_on='METRIC',
    right_on='display_name',
    how='left'
 )
if metrics['id'].isna().any():
    unknown = metrics.loc[metrics['id'].isna(), 'METRIC'].drop_duplicates().tolist()
    raise ValueError(f'Metricas de dataset fuera de catalogo: {unknown}')

catalog['validation_status'] = catalog['validation_status'].astype(str)
catalog['direction_confidence'] = catalog['direction_confidence'].astype(str)

suspended_metrics = set(catalog.loc[catalog['validation_status'].eq('suspended_pending_definition'), 'id'])
peer_restricted_metrics = set(
    x.get('metric_id') for x in business_rules.get('peer_groups', {}).get('restricted_metrics', []) if isinstance(x, dict)
 )
peer_restricted_metrics |= set(catalog.loc[catalog['low_coverage_peer_groups'].fillna(False), 'id'])

print('metrics_long:', metrics_long.shape)
print('orders_long :', orders_long.shape)
print('zone_master :', zone_master.shape)
print('catalog metrics:', len(catalog))
print('suspended_metrics:', sorted(suspended_metrics))
print('peer_restricted_metrics:', sorted(peer_restricted_metrics))

metrics_long: (104490, 14)
orders_long : (11178, 11)
zone_master : (1244, 13)
catalog metrics: 13
suspended_metrics: ['lead_penetration']
peer_restricted_metrics: ['lead_penetration', 'turbo_adoption']


In [4]:
semantic_checks = semantic_report.get('semantic_checks', {})
checks_df = pd.DataFrame([
    {'check': k, 'status': v.get('status', 'UNKNOWN'), 'detail': v.get('detail', '')}
    for k, v in semantic_checks.items()
]).sort_values('check')

if checks_df.empty:
    checks_df = pd.DataFrame([{'check': 'semantic_checks_missing', 'status': 'WARN', 'detail': 'No se encontro semantic_layer_report.json'}])

n_fail = int((checks_df['status'] == 'FAIL').sum())
n_warn = int((checks_df['status'] == 'WARN').sum())

rules_presence = pd.DataFrame([
    {'rule': 'peer_group_primary', 'available': 'primary' in business_rules.get('peer_groups', {})},
    {'rule': 'peer_group_size_rules', 'available': 'size_rules' in business_rules.get('peer_groups', {})},
    {'rule': 'fallback_behavior', 'available': 'fallback_behavior' in business_rules.get('peer_groups', {})},
    {'rule': 'insight_detectors', 'available': 'insight_detectors' in business_rules},
    {'rule': 'temporal_rules', 'available': 'temporal' in business_rules},
])

display(checks_df)
display(rules_presence)
print({'semantic_FAIL': n_fail, 'semantic_WARN': n_warn})

,check,status,detail
0,all_dataset_metrics_in_catalog,PASS,"dataset=13, catalog=13, missing=set()"
4,all_metrics_have_confidence,PASS,all entries have direction_confidence field
3,all_metrics_have_direction,PASS,all entries have desired_direction field
8,intents_have_examples,PASS,
7,intents_have_required_params,PASS,n_intents=8
2,metric_ids_are_unique,PASS,"n_ids=13, n_unique=13"
1,no_orphan_metrics_in_catalog,PASS,orphans=set()
6,peer_groups_have_reliable_groups,PASS,"reliable_groups=25, total_groups=43"
9,scale_violations_documented,PASS,violations=none
5,zone_key_is_unique,PASS,n_zone_keys=1244


,rule,available
0,peer_group_primary,True
1,peer_group_size_rules,True
2,fallback_behavior,True
3,insight_detectors,True
4,temporal_rules,True


{'semantic_FAIL': 0, 'semantic_WARN': 0}


### Que nos llevamos de esta seccion

- El engine solo corre sobre metricas presentes en el catalogo gobernado.
- Las metricas suspendidas quedan identificadas desde el inicio.
- Se verifica disponibilidad de reglas de peers, temporalidad y detectores.
- Cualquier FAIL semantico queda visible para tratamiento explicito.

## 2) Marco de detectores

### Detectores implementados
1. `anomaly_point`: desviacion puntual robusta + WoW material.
2. `persistent_deterioration`: deterioro sostenido via rachas de empeoramiento.
3. `peer_gap`: brecha contra mediana de peer group gobernado.
4. `opportunity`: mejora consistente y/o outperformance defendible.
5. `possible_driver`: asociacion exploratoria prudente (no causalidad).

### Que detectan y que no detectan
- Detectan señales candidatas para investigacion operativa.
- No prueban causalidad.
- No reemplazan validacion de negocio.

### Por que son defendibles con 9 semanas
- Priorizan reglas simples y trazables.
- Usan estadistica robusta (MAD, medianas) en vez de supuestos fuertes.
- Incorporan caveats de confianza, cobertura y validacion semantica.

### Riesgos abiertos
- Thresholds provisionales.
- Historico corto para persistencia y asociaciones.
- Peer groups pequenos en parte del universo.

In [5]:
temporal_cfg = business_rules.get('temporal', {})
det_cfg = business_rules.get('insight_detectors', {})
peer_cfg = business_rules.get('peer_groups', {})

current_offset = temporal_cfg.get('current_week_offset', 'L0W')
min_history_trend = int(temporal_cfg.get('min_history_for_trend_analysis', 4))
min_history_streak = int(temporal_cfg.get('min_history_for_streak_detection', 3))

wow_threshold_pct = float(det_cfg.get('wow_delta', {}).get('alert_threshold_pct', 0.10))
streak_min_weeks = int(det_cfg.get('decline_streak', {}).get('min_weeks_for_alert', 3))
robust_z_threshold = float(det_cfg.get('robust_z_score', {}).get('anomaly_threshold', 2.5))

min_peer_size = int(peer_cfg.get('size_rules', {}).get('min_size_for_benchmark', 10))
min_peer_warn = int(peer_cfg.get('size_rules', {}).get('min_size_warning_threshold', 5))
peer_dims = peer_cfg.get('primary', {}).get('dimensions', ['COUNTRY', 'ZONE_TYPE', 'ZONE_PRIORITIZATION'])

priority_map = {
    'High Priority': 1.0,
    'Prioritized': 0.75,
    'Not Prioritized': 0.45,
}

def metric_direction_multiplier(direction: str) -> float:
    if direction == 'higher_is_better':
        return 1.0
    if direction == 'lower_is_better':
        return -1.0
    return np.nan

def build_entity_keys(row: pd.Series) -> str:
    return json.dumps({'COUNTRY': row['COUNTRY'], 'CITY': row['CITY'], 'ZONE': row['ZONE']}, ensure_ascii=False)

def build_display_entity(row: pd.Series) -> str:
    return f"{row['COUNTRY']} | {row['CITY']} | {row['ZONE']}"

def base_caveats(row: pd.Series) -> list[str]:
    c = []
    if str(row.get('direction_confidence', '')).lower() == 'provisional':
        c.append('direction provisional')
    if str(row.get('validation_status', '')).lower() != 'confirmed':
        c.append(f"validation_status={row.get('validation_status')}")
    return c

def compute_business_priority(zone_prioritization: str) -> float:
    return float(priority_map.get(str(zone_prioritization), 0.50))

def compute_confidence(row: pd.Series, base: float = 1.0, peer_confidence: str | None = None, evidence_quality: float = 1.0) -> float:
    score = float(base)
    if str(row.get('direction_confidence', '')) == 'provisional':
        score *= 0.85
    if str(row.get('validation_status', '')) == 'pending_business_validation':
        score *= 0.85
    if str(row.get('validation_status', '')) == 'suspended_pending_definition':
        score *= 0.40
    if peer_confidence == 'low_confidence':
        score *= 0.75
    elif peer_confidence == 'too_small':
        score *= 0.45
    score *= float(evidence_quality)
    return float(np.clip(score, 0.0, 1.0))

def to_json_struct(payload: dict) -> str:
    return json.dumps(payload, ensure_ascii=False, sort_keys=True)

print({
    'current_offset': current_offset,
    'wow_threshold_pct': wow_threshold_pct,
    'streak_min_weeks': streak_min_weeks,
    'robust_z_threshold': robust_z_threshold,
    'min_peer_size': min_peer_size,
    'min_peer_warn': min_peer_warn,
    'peer_dims': peer_dims,
})

{'current_offset': 'L0W', 'wow_threshold_pct': 0.1, 'streak_min_weeks': 3, 'robust_z_threshold': 2.5, 'min_peer_size': 10, 'min_peer_warn': 5, 'peer_dims': ['COUNTRY', 'ZONE_TYPE', 'ZONE_PRIORITIZATION']}


## 3) Detector de anomalia puntual (`anomaly_point`)

### Metodo
- Baseline robusto por entidad-metrca: mediana y MAD sobre 9 semanas.
- Puntaje de anomalia: modified z-score.
- Materialidad: exige cambio WoW minimo por threshold provisional.
- Conciencia de direccion: solo alerta si el cambio es desfavorable segun `desired_direction`.

### Limite
No prueba causa; detecta desviaciones candidatas para investigacion.

In [6]:
entity_cols = ['COUNTRY', 'CITY', 'ZONE']
group_cols = entity_cols + ['id']

stats = (
    metrics.groupby(group_cols, dropna=False)['VALUE']
    .agg(
        median_value='median',
        mad=lambda s: float(np.nanmedian(np.abs(s - np.nanmedian(s)))) if s.notna().any() else np.nan,
        history_points=lambda s: int(s.notna().sum()),
    )
    .reset_index()
 )

l0 = metrics.loc[metrics['WEEK_OFFSET'].eq(current_offset)].copy()
current_num = int(l0['week_offset_num'].iloc[0]) if not l0.empty else 0
l1 = metrics.loc[metrics['week_offset_num'].eq(current_num + 1), entity_cols + ['id', 'VALUE']].rename(columns={'VALUE': 'prev_value'})

latest = l0.merge(l1, on=group_cols, how='left').merge(stats, on=group_cols, how='left')
latest['raw_value'] = latest['VALUE']
latest['wow_delta'] = latest['raw_value'] - latest['prev_value']
latest['wow_pct'] = np.where(latest['prev_value'].abs() > 1e-12, latest['wow_delta'] / latest['prev_value'], np.nan)
latest['anomaly_score'] = np.where(
    latest['mad'] > 0,
    0.6745 * (latest['raw_value'] - latest['median_value']) / latest['mad'],
    np.nan,
 )
latest['direction_mult'] = latest['desired_direction'].map(metric_direction_multiplier)
latest['oriented_wow'] = latest['wow_delta'] * latest['direction_mult']
latest['worsening_magnitude'] = (-latest['oriented_wow']).clip(lower=0.0)

anomaly_mask = (
    ~latest['id'].isin(suspended_metrics)
    & latest['history_points'].ge(min_history_trend)
    & latest['worsening_magnitude'].gt(0)
    & latest['anomaly_score'].abs().ge(robust_z_threshold)
    & latest['wow_pct'].abs().ge(wow_threshold_pct)
)

anomaly_candidates = latest.loc[anomaly_mask].copy()
anomaly_candidates['severity_score'] = np.clip(
    0.55 * (anomaly_candidates['anomaly_score'].abs() / robust_z_threshold)
    + 0.45 * (anomaly_candidates['wow_pct'].abs() / wow_threshold_pct),
    0,
    2,
) / 2
anomaly_candidates['confidence_score'] = anomaly_candidates.apply(
    lambda r: compute_confidence(r, base=0.95, evidence_quality=1.0 if r['history_points'] >= 6 else 0.8),
    axis=1,
 )
anomaly_candidates['business_priority_score'] = anomaly_candidates['ZONE_PRIORITIZATION'].map(compute_business_priority)
anomaly_candidates['insight_category'] = 'anomaly'
anomaly_candidates['detector_name'] = 'anomaly_point'
anomaly_candidates['metric_id'] = anomaly_candidates['id']
anomaly_candidates['metric_display_name'] = anomaly_candidates['display_name']
anomaly_candidates['time_context'] = current_offset
anomaly_candidates['entity_keys'] = anomaly_candidates.apply(build_entity_keys, axis=1)
anomaly_candidates['display_entity'] = anomaly_candidates.apply(build_display_entity, axis=1)
anomaly_candidates['peer_group_info'] = ''
anomaly_candidates['evidence_struct'] = anomaly_candidates.apply(
    lambda r: to_json_struct({
        'raw_value': float(r['raw_value']) if pd.notna(r['raw_value']) else None,
        'prev_value': float(r['prev_value']) if pd.notna(r['prev_value']) else None,
        'wow_delta': float(r['wow_delta']) if pd.notna(r['wow_delta']) else None,
        'wow_pct': float(r['wow_pct']) if pd.notna(r['wow_pct']) else None,
        'median_value': float(r['median_value']) if pd.notna(r['median_value']) else None,
        'mad': float(r['mad']) if pd.notna(r['mad']) else None,
        'modified_z': float(r['anomaly_score']) if pd.notna(r['anomaly_score']) else None,
        'thresholds': {'robust_z': robust_z_threshold, 'wow_pct': wow_threshold_pct},
    }),
    axis=1,
 )
anomaly_candidates['caveats'] = anomaly_candidates.apply(
    lambda r: '; '.join(base_caveats(r) + ['thresholds provisionales']),
    axis=1,
 )

display(anomaly_candidates[['display_entity', 'metric_id', 'raw_value', 'wow_delta', 'anomaly_score', 'severity_score', 'confidence_score']].head(10))
print('anomaly_candidates:', anomaly_candidates.shape)

,display_entity,metric_id,raw_value,wow_delta,anomaly_score,severity_score,confidence_score
18,PE | Lima | San isidro,gross_profit_ue,2.553866,-0.753749,-4.745618,1.0,0.686375
20,PE | Lima | Monterrico,gross_profit_ue,2.051294,-0.914593,-4.944146,1.0,0.686375
64,AR | Cordoba | CORDOBA_COLON,gross_profit_ue,-0.652380,-0.786277,-2.952539,1.0,0.686375
92,PE | Arequipa | PichuPichu Este,gross_profit_ue,-0.911428,-1.057164,-3.151519,1.0,0.686375
115,PE | Lima | San Borja,gross_profit_ue,2.340620,-0.909879,-6.072142,1.0,0.686375
133,CO | Medellin | Poblado_guayabal,gross_profit_ue,2.687869,-0.410770,-7.110233,1.0,0.686375
135,PE | Piura | Enace,gross_profit_ue,-0.472474,-1.080161,-2.543306,1.0,0.686375
154,BR | Florianopolis | SC - FLN - São José,gross_profit_ue,-13.551081,-10.007791,-4.055341,1.0,0.686375
160,MX | Monterrey | MTY_Apodaca_Huinalá,gross_profit_ue,0.132192,-0.128137,24.921579,1.0,0.686375
190,AR | La Plata | City Bell,gross_profit_ue,-2.822862,-1.775655,-2.927171,1.0,0.686375


anomaly_candidates: (142, 53)


## 4) Detector de deterioro persistente (`persistent_deterioration`)

### Metodo
Se mide deterioro sostenido por rachas consecutivas de cambio desfavorable WoW, respetando `desired_direction`.

### Diferencia frente a anomalia puntual
- Anomalia puntual: evento abrupto.
- Deterioro persistente: patron sostenido en el tiempo.

In [7]:
def compute_streaks(group: pd.DataFrame) -> pd.Series:
    g = group.sort_values('week_offset_num', ascending=True).copy()
    direction = g['desired_direction'].iloc[0]
    mult = metric_direction_multiplier(direction)
    values = g['VALUE'].to_numpy(dtype=float)
    if len(values) < 2:
        return pd.Series({'streak_deterioration': 0, 'streak_improvement': 0, 'worsening_share_last4': np.nan})

    diffs = values[:-1] - values[1:]
    oriented = diffs * mult

    det = 0
    for v in oriented:
        if pd.isna(v) or v >= 0:
            break
        det += 1

    imp = 0
    for v in oriented:
        if pd.isna(v) or v <= 0:
            break
        imp += 1

    tail = oriented[:4] if len(oriented) >= 4 else oriented
    worsening_share_last4 = float(np.mean(tail < 0)) if len(tail) > 0 else np.nan

    return pd.Series({
        'streak_deterioration': int(det),
        'streak_improvement': int(imp),
        'worsening_share_last4': worsening_share_last4,
    })

trend_summary = (
    metrics.groupby(group_cols, dropna=False)
    .apply(compute_streaks)
    .reset_index()
 )

deterioration = latest.merge(trend_summary, on=group_cols, how='left')
det_mask = (
    ~deterioration['id'].isin(suspended_metrics)
    & deterioration['history_points'].ge(min_history_streak)
    & deterioration['streak_deterioration'].ge(streak_min_weeks)
)

deterioration_candidates = deterioration.loc[det_mask].copy()
deterioration_candidates['deterioration_score'] = np.clip(
    0.7 * (deterioration_candidates['streak_deterioration'] / streak_min_weeks)
    + 0.3 * deterioration_candidates['worsening_share_last4'].fillna(0),
    0,
    2,
) / 2
deterioration_candidates['severity_score'] = deterioration_candidates['deterioration_score']
deterioration_candidates['confidence_score'] = deterioration_candidates.apply(
    lambda r: compute_confidence(r, base=0.90, evidence_quality=1.0 if r['history_points'] >= 6 else 0.75),
    axis=1,
 )
deterioration_candidates['business_priority_score'] = deterioration_candidates['ZONE_PRIORITIZATION'].map(compute_business_priority)
deterioration_candidates['insight_category'] = 'persistent_deterioration'
deterioration_candidates['detector_name'] = 'persistent_deterioration'
deterioration_candidates['metric_id'] = deterioration_candidates['id']
deterioration_candidates['metric_display_name'] = deterioration_candidates['display_name']
deterioration_candidates['time_context'] = current_offset
deterioration_candidates['entity_keys'] = deterioration_candidates.apply(build_entity_keys, axis=1)
deterioration_candidates['display_entity'] = deterioration_candidates.apply(build_display_entity, axis=1)
deterioration_candidates['peer_group_info'] = ''
deterioration_candidates['evidence_struct'] = deterioration_candidates.apply(
    lambda r: to_json_struct({
        'streak_length': int(r['streak_deterioration']),
        'worsening_share_last4': float(r['worsening_share_last4']) if pd.notna(r['worsening_share_last4']) else None,
        'wow_delta_current': float(r['wow_delta']) if pd.notna(r['wow_delta']) else None,
        'streak_threshold': streak_min_weeks,
    }),
    axis=1,
 )
deterioration_candidates['caveats'] = deterioration_candidates.apply(
    lambda r: '; '.join(base_caveats(r) + ['streak threshold provisional']),
    axis=1,
 )

display(deterioration_candidates[['display_entity', 'metric_id', 'streak_deterioration', 'deterioration_score', 'severity_score', 'confidence_score']].head(10))
print('deterioration_candidates:', deterioration_candidates.shape)

,display_entity,metric_id,streak_deterioration,deterioration_score,severity_score,confidence_score
14,AR | Cordoba | General Paz,retail_sst_ss_cvr,4.0,0.616667,0.616667,0.65025
44,EC | Santa Elena / Salinas / Libertad | Salinas,retail_sst_ss_cvr,3.0,0.462500,0.462500,0.65025
48,AR | Buenos Aires | ESCOBAR,retail_sst_ss_cvr,7.0,0.966667,0.966667,0.65025
55,BR | Jundiai | Varzea Paulista,gross_profit_ue,4.0,0.616667,0.616667,0.65025
59,BR | Brasilia | DF - BSB - Asa Sul/Asa Norte,retail_sst_ss_cvr,4.0,0.616667,0.616667,0.65025
70,BR | Porto Alegre | RS - POA - Sarandi,gross_profit_ue,3.0,0.462500,0.462500,0.65025
116,AR | Rio Cuarto | RIO_CUARTO,gross_profit_ue,4.0,0.616667,0.616667,0.65025
133,CO | Medellin | Poblado_guayabal,gross_profit_ue,3.0,0.462500,0.462500,0.65025
158,BR | Porto Alegre | RS - POA - Navegantes,retail_sst_ss_cvr,5.0,0.733333,0.733333,0.65025
183,AR | San Salvador De Jujuy | JUJUY,retail_sst_ss_cvr,5.0,0.733333,0.733333,0.65025


deterioration_candidates: (1399, 57)


## 5) Detector de brecha contra peers (`peer_gap`)

### Metodo
- Peer group primario: `(COUNTRY, ZONE_TYPE, ZONE_PRIORITIZATION)`.
- Baseline: mediana del peer en L0W.
- Clasificacion de confianza del grupo:
  - `reliable` si `n >= min_size_for_benchmark`
  - `low_confidence` si `min_size_warning_threshold <= n < min_size_for_benchmark`
  - `too_small` si `n < min_size_warning_threshold` (se excluye del output de benchmark).

### Restricciones criticas
- `lead_penetration` y metricas restringidas no entran a benchmark.
- `turbo_adoption` se excluye por cobertura.
- `lower_is_better` invierte la lectura de la brecha.

In [8]:
peer_base = l0.loc[~l0['id'].isin(peer_restricted_metrics)].copy()
peer_group_cols = peer_dims + ['id']

peer_stats = (
    peer_base.groupby(peer_group_cols, dropna=False)['VALUE']
    .agg(peer_group_size='size', peer_median='median', peer_p25=lambda s: s.quantile(0.25), peer_p75=lambda s: s.quantile(0.75))
    .reset_index()
 )

peer_eval = peer_base.merge(peer_stats, on=peer_group_cols, how='left')
peer_eval['peer_confidence'] = np.select(
    [
        peer_eval['peer_group_size'] >= min_peer_size,
        (peer_eval['peer_group_size'] >= min_peer_warn) & (peer_eval['peer_group_size'] < min_peer_size),
        peer_eval['peer_group_size'] < min_peer_warn,
    ],
    ['reliable', 'low_confidence', 'too_small'],
    default='unknown',
 )
peer_eval['metric_id'] = peer_eval['id']
peer_eval['entity_value'] = peer_eval['VALUE']
peer_eval['gap_abs'] = peer_eval['entity_value'] - peer_eval['peer_median']
peer_eval['gap_pct'] = np.where(peer_eval['peer_median'].abs() > 1e-12, peer_eval['gap_abs'] / peer_eval['peer_median'], np.nan)
peer_eval['direction_mult'] = peer_eval['desired_direction'].map(metric_direction_multiplier)
peer_eval['oriented_gap'] = peer_eval['gap_abs'] * peer_eval['direction_mult']
peer_eval['underperforming'] = peer_eval['oriented_gap'] < 0

peer_mask = (
    ~peer_eval['metric_id'].isin(suspended_metrics)
    & peer_eval['peer_confidence'].isin(['reliable', 'low_confidence'])
    & peer_eval['underperforming']
    & peer_eval['gap_pct'].abs().ge(wow_threshold_pct)
)

peer_candidates = peer_eval.loc[peer_mask].copy()
peer_candidates['severity_score'] = np.clip(peer_candidates['gap_pct'].abs() / wow_threshold_pct, 0, 2) / 2
peer_candidates['confidence_score'] = peer_candidates.apply(
    lambda r: compute_confidence(r, base=0.90, peer_confidence=str(r['peer_confidence']), evidence_quality=0.95),
    axis=1,
 )
peer_candidates['business_priority_score'] = peer_candidates['ZONE_PRIORITIZATION'].map(compute_business_priority)
peer_candidates['insight_category'] = 'peer_gap'
peer_candidates['detector_name'] = 'peer_gap'
peer_candidates['metric_display_name'] = peer_candidates['display_name']
peer_candidates['time_context'] = current_offset
peer_candidates['entity_keys'] = peer_candidates.apply(build_entity_keys, axis=1)
peer_candidates['display_entity'] = peer_candidates.apply(build_display_entity, axis=1)
peer_candidates['peer_group_info'] = peer_candidates.apply(
    lambda r: to_json_struct({
        'dimensions': peer_dims,
        'COUNTRY': r['COUNTRY'],
        'ZONE_TYPE': r.get('ZONE_TYPE'),
        'ZONE_PRIORITIZATION': r.get('ZONE_PRIORITIZATION'),
        'peer_group_size': int(r['peer_group_size']) if pd.notna(r['peer_group_size']) else None,
        'peer_confidence': r['peer_confidence'],
    }),
    axis=1,
 )
peer_candidates['evidence_struct'] = peer_candidates.apply(
    lambda r: to_json_struct({
        'entity_value': float(r['entity_value']) if pd.notna(r['entity_value']) else None,
        'peer_median': float(r['peer_median']) if pd.notna(r['peer_median']) else None,
        'gap_abs': float(r['gap_abs']) if pd.notna(r['gap_abs']) else None,
        'gap_pct': float(r['gap_pct']) if pd.notna(r['gap_pct']) else None,
        'peer_confidence': r['peer_confidence'],
    }),
    axis=1,
 )
peer_candidates['caveats'] = peer_candidates.apply(
    lambda r: '; '.join(
        base_caveats(r)
        + ([f"peer_confidence={r['peer_confidence']}"] if r['peer_confidence'] != 'reliable' else [])
        + ['benchmark con reglas de comparabilidad NB20']
    ),
    axis=1,
 )

display(peer_candidates[['display_entity', 'metric_id', 'peer_group_size', 'peer_confidence', 'gap_abs', 'gap_pct', 'severity_score', 'confidence_score']].head(10))
print('peer_candidates:', peer_candidates.shape)

,display_entity,metric_id,peer_group_size,peer_confidence,gap_abs,gap_pct,severity_score,confidence_score
0,EC | Quito | San Martin de Porras,retail_sst_ss_cvr,28,reliable,-0.106247,-0.126214,0.631071,0.617737
1,BR | Bauru | FS Parque Jardim Europa/Vila Riac...,restaurants_sst_ss_cvr,135,reliable,-0.367102,-0.435935,1.000000,0.617737
9,BR | Recife | PE - REC - Zona Norte,gross_profit_ue,12,reliable,-2.065673,-0.558165,1.000000,0.617737
11,BR | Grande São Paulo | SP - GRU - Centro,gross_profit_ue,133,reliable,-1.175004,2.373302,1.000000,0.617737
13,CO | Valledupar | San Jorge,gross_profit_ue,38,reliable,-0.690509,-0.343435,1.000000,0.617737
21,PE | Huancayo | Huancayo centro,gross_profit_ue,39,reliable,-0.782712,1.141386,1.000000,0.617737
23,MX | Dolores Hidalgo | DLH Dolores Hidalgo,gross_profit_ue,93,reliable,-0.858256,-0.576073,1.000000,0.617737
31,MX | Aguascalientes | AGS_Jesus Maria,gross_profit_ue,119,reliable,-0.374600,-0.204470,1.000000,0.617737
32,MX | Ciudad Guzman | CDG La Hacienda,gross_profit_ue,93,reliable,-15.130151,-10.155562,1.000000,0.617737
34,AR | Cordoba | Villa Adela,gross_profit_ue,15,reliable,-2.641900,-1.090464,1.000000,0.617737


peer_candidates: (2321, 53)


## 6) Detector de oportunidades (`opportunity`)

### Metodo
Una oportunidad se marca cuando hay evidencia positiva y sostenible, no solo ruido puntual.

Reglas activas:
- mejora consistente (racha de mejora), o
- outperforming frente a peers con confianza aceptable.

### Limites
Es una señal de priorizacion, no una recomendacion causal cerrada.

In [9]:
opportunities_base = latest.merge(trend_summary, on=group_cols, how='left')
peer_for_opps = peer_eval[[*group_cols, 'peer_group_size', 'peer_confidence', 'peer_median', 'gap_abs', 'gap_pct', 'oriented_gap']].copy()
opportunities_base = opportunities_base.merge(peer_for_opps, on=group_cols, how='left')

opportunities_base['is_improving_streak'] = opportunities_base['streak_improvement'].fillna(0).ge(streak_min_weeks)
opportunities_base['is_peer_outperform'] = (
    opportunities_base['peer_confidence'].isin(['reliable', 'low_confidence'])
    & opportunities_base['oriented_gap'].gt(0)
    & opportunities_base['gap_pct'].abs().ge(wow_threshold_pct)
)
opportunities_base['is_priority_zone'] = opportunities_base['ZONE_PRIORITIZATION'].eq('High Priority')

opp_mask = (
    ~opportunities_base['id'].isin(suspended_metrics)
    & (opportunities_base['is_improving_streak'] | opportunities_base['is_peer_outperform'])
)

opportunity_candidates = opportunities_base.loc[opp_mask].copy()
opportunity_candidates['opportunity_score'] = np.clip(
    0.6 * np.where(opportunity_candidates['is_improving_streak'], opportunity_candidates['streak_improvement'] / streak_min_weeks, 0)
    + 0.4 * np.where(opportunity_candidates['is_peer_outperform'], opportunity_candidates['gap_pct'].abs() / wow_threshold_pct, 0),
    0,
    2,
) / 2
opportunity_candidates['severity_score'] = opportunity_candidates['opportunity_score']
opportunity_candidates['confidence_score'] = opportunity_candidates.apply(
    lambda r: compute_confidence(
        r,
        base=0.88,
        peer_confidence=str(r['peer_confidence']) if pd.notna(r['peer_confidence']) else None,
        evidence_quality=1.0 if r['is_improving_streak'] else 0.9,
    ),
    axis=1,
 )
opportunity_candidates['business_priority_score'] = opportunity_candidates['ZONE_PRIORITIZATION'].map(compute_business_priority)
opportunity_candidates['insight_category'] = 'opportunity'
opportunity_candidates['detector_name'] = 'opportunity'
opportunity_candidates['metric_id'] = opportunity_candidates['id']
opportunity_candidates['metric_display_name'] = opportunity_candidates['display_name']
opportunity_candidates['time_context'] = current_offset
opportunity_candidates['entity_keys'] = opportunity_candidates.apply(build_entity_keys, axis=1)
opportunity_candidates['display_entity'] = opportunity_candidates.apply(build_display_entity, axis=1)
opportunity_candidates['peer_group_info'] = opportunity_candidates.apply(
    lambda r: to_json_struct({
        'peer_group_size': int(r['peer_group_size']) if pd.notna(r['peer_group_size']) else None,
        'peer_confidence': r['peer_confidence'] if pd.notna(r['peer_confidence']) else None,
    }),
    axis=1,
 )
opportunity_candidates['evidence_struct'] = opportunity_candidates.apply(
    lambda r: to_json_struct({
        'streak_improvement': int(r['streak_improvement']) if pd.notna(r['streak_improvement']) else 0,
        'is_peer_outperform': bool(r['is_peer_outperform']),
        'gap_pct': float(r['gap_pct']) if pd.notna(r['gap_pct']) else None,
        'high_priority_zone': bool(r['is_priority_zone']),
    }),
    axis=1,
 )
opportunity_candidates['caveats'] = opportunity_candidates.apply(
    lambda r: '; '.join(
        base_caveats(r)
        + ([f"peer_confidence={r['peer_confidence']}"] if pd.notna(r['peer_confidence']) and r['peer_confidence'] != 'reliable' else [])
        + ['oportunidad basada en reglas, no causalidad']
    ),
    axis=1,
 )

display(opportunity_candidates[['display_entity', 'metric_id', 'streak_improvement', 'is_peer_outperform', 'severity_score', 'confidence_score']].head(10))
print('opportunity_candidates:', opportunity_candidates.shape)

,display_entity,metric_id,streak_improvement,is_peer_outperform,severity_score,confidence_score
2,MX | Mexicali | MXL_Universidad,retail_sst_ss_cvr,3.0,False,0.300000,0.635800
4,MX | Guadalajara | Valle Real,gross_profit_ue,0.0,True,1.000000,0.572220
7,CO | Cartagena | Bocagrande,retail_sst_ss_cvr,6.0,False,0.600000,0.635800
8,BR | Campinas | SP - VCP - Jardim Itatinga,gross_profit_ue,4.0,True,1.000000,0.635800
10,MX | Nuevo Laredo | Nlar_centro,gross_profit_ue,0.0,True,1.000000,0.572220
12,MX | Monterrey | Mitras Centro,retail_sst_ss_cvr,4.0,False,0.400000,0.635800
16,MX | Cuernavaca | CVA-Centro,gross_profit_ue,6.0,True,1.000000,0.635800
17,PE | Trujillo | Centro Trujillo,retail_sst_ss_cvr,3.0,False,0.300000,0.476850
18,PE | Lima | San isidro,gross_profit_ue,0.0,True,0.495691,0.429165
19,BR | Grande São Paulo | Jardins/Pinheiros,gross_profit_ue,0.0,True,0.395781,0.572220


opportunity_candidates: (3270, 66)


## 7) Modulo de posibles drivers asociados (`possible_driver`)

### Metodo
Se evalua asociacion monotona por zona entre `orders` y candidatas via Spearman WoW.

### Regla critica
- Se reporta como `possible_driver`, `weak_hypothesis` o `hypothesis`.
- Nunca se interpreta como causalidad.
- Si la evidencia es debil o n pequeno, se degrada confianza y lenguaje.

In [10]:
orders = orders_long.copy()
orders['metric_id'] = 'orders_total'

orders_wow = orders.sort_values(entity_cols + ['week_offset_num'], ascending=[True, True, True, True]).copy()
orders_wow['orders_wow_pct'] = orders_wow.groupby(entity_cols, dropna=False)['VALUE'].pct_change(-1)
orders_wow = orders_wow[entity_cols + ['week_offset_num', 'orders_wow_pct']]

metrics_wow = metrics.sort_values(group_cols + ['week_offset_num'], ascending=[True, True, True, True, True]).copy()
metrics_wow['metric_wow_pct'] = metrics_wow.groupby(group_cols, dropna=False)['VALUE'].pct_change(-1)
metrics_wow = metrics_wow[entity_cols + ['id', 'display_name', 'validation_status', 'direction_confidence', 'week_offset_num', 'metric_wow_pct']]

driver_base = metrics_wow.merge(orders_wow, on=entity_cols + ['week_offset_num'], how='inner')
driver_base = driver_base.replace([np.inf, -np.inf], np.nan).dropna(subset=['metric_wow_pct', 'orders_wow_pct'])
driver_base = driver_base.loc[~driver_base['id'].isin(suspended_metrics)].copy()

def zone_metric_assoc(g: pd.DataFrame) -> pd.Series:
    n = int(g.shape[0])
    if n < 5:
        return pd.Series({'n_points': n, 'spearman_rho': np.nan, 'p_value': np.nan})
    rho, p = spearmanr(g['metric_wow_pct'], g['orders_wow_pct'], nan_policy='omit')
    return pd.Series({'n_points': n, 'spearman_rho': float(rho), 'p_value': float(p)})

driver_assoc = (
    driver_base.groupby(entity_cols + ['id', 'display_name', 'validation_status', 'direction_confidence'], dropna=False)
    .apply(zone_metric_assoc)
    .reset_index()
 )
driver_assoc = driver_assoc.dropna(subset=['spearman_rho'])
driver_assoc['abs_rho'] = driver_assoc['spearman_rho'].abs()

driver_candidates = driver_assoc.loc[driver_assoc['abs_rho'] >= 0.25].copy()
driver_candidates['narrative_label'] = np.select(
    [
        driver_candidates['abs_rho'] >= 0.50,
        driver_candidates['abs_rho'] >= 0.35,
    ],
    ['hypothesis', 'possible_driver'],
    default='weak_hypothesis',
 )
driver_candidates['insight_category'] = 'possible_driver'
driver_candidates['detector_name'] = 'possible_driver'
driver_candidates['metric_id'] = 'orders_total'
driver_candidates['metric_display_name'] = 'Orders'
driver_candidates['target_metric'] = 'orders_total'
driver_candidates['candidate_driver'] = driver_candidates['id']
driver_candidates['time_context'] = 'L8W-L0W (WoW associations)'
driver_candidates['severity_score'] = np.clip(driver_candidates['abs_rho'], 0, 1)
driver_candidates['confidence_score'] = driver_candidates.apply(
    lambda r: compute_confidence(
        r,
        base=0.78,
        evidence_quality=0.80 if r['n_points'] < 7 else 1.0,
    ) * (0.9 if r['p_value'] > 0.05 else 1.0),
    axis=1,
 )
driver_candidates['business_priority_score'] = 0.50
driver_candidates['entity_keys'] = driver_candidates.apply(build_entity_keys, axis=1)
driver_candidates['display_entity'] = driver_candidates.apply(build_display_entity, axis=1)
driver_candidates['peer_group_info'] = ''
driver_candidates['evidence_struct'] = driver_candidates.apply(
    lambda r: to_json_struct({
        'evidence_type': 'spearman_wow_zone',
        'association_strength': float(r['spearman_rho']),
        'n_points': int(r['n_points']),
        'p_value': float(r['p_value']) if pd.notna(r['p_value']) else None,
        'label': r['narrative_label'],
        'non_causal': True,
    }),
    axis=1,
 )
driver_candidates['caveats'] = driver_candidates.apply(
    lambda r: '; '.join(base_caveats(r) + ['association_not_causation', 'history_short_9_weeks']),
    axis=1,
 )

display(driver_candidates[['display_entity', 'candidate_driver', 'spearman_rho', 'p_value', 'narrative_label', 'confidence_score']].head(10))
print('driver_candidates:', driver_candidates.shape)

,display_entity,candidate_driver,spearman_rho,p_value,narrative_label,confidence_score
0,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,gross_profit_ue,0.261905,0.530923,weak_hypothesis,0.507195
1,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,mltv_top_verticals_adoption,-0.333333,0.419753,weak_hypothesis,0.507195
3,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,pct_pro_users_breakeven,-0.476190,0.232936,possible_driver,0.507195
4,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,pct_restaurants_sessions_optimal_assortment,0.690476,0.057990,hypothesis,0.507195
6,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,pro_adoption_last_week,0.642857,0.085559,hypothesis,0.507195
7,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,restaurants_markdowns_gmv,0.380952,0.351813,possible_driver,0.507195
8,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,restaurants_ss_atc_cvr,0.714286,0.046528,hypothesis,0.563550
9,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,restaurants_sst_ss_cvr,-0.357143,0.385121,possible_driver,0.507195
10,AR | Bahia Blanca | BAHIA_BLANCA_NORTE,retail_sst_ss_cvr,-0.261905,0.530923,weak_hypothesis,0.507195
11,AR | Bahia Blanca | BAHIA_BLANCA_SUR,gross_profit_ue,0.571429,0.138960,hypothesis,0.507195


driver_candidates: (5311, 27)


## 8) Sistema de scoring y priorizacion

### Logica de score
- `severity_score`: intensidad de la senal del detector.
- `confidence_score`: confianza segun calidad de evidencia + gobernanza semantica.
- `business_priority_score`: prioridad operativa de la zona.
- `final_rank_score`: combinacion ponderada + penalizaciones explicitas.

### Penalizaciones
- `direction_confidence = provisional`
- `validation_status != confirmed`
- `peer_confidence = low_confidence` (si aplica)

Los pesos son configurables y se reportan como provisionales.

In [11]:
def ensure_columns(df: pd.DataFrame, extra_defaults: dict | None = None) -> pd.DataFrame:
    defaults = {
        'target_metric': None,
        'candidate_driver': None,
        'raw_value': np.nan,
        'wow_delta': np.nan,
        'anomaly_score': np.nan,
        'deterioration_score': np.nan,
        'streak_length': np.nan,
        'peer_group_description': None,
        'peer_group_size': np.nan,
        'peer_confidence': None,
        'entity_value': np.nan,
        'peer_median': np.nan,
        'gap_abs': np.nan,
        'gap_pct': np.nan,
        'narrative_label': None,
        'association_strength': np.nan,
        'final_rank_score': np.nan,
        'narrative_stub': '',
        'recommendation_stub': '',
    }
    if extra_defaults:
        defaults.update(extra_defaults)
    out = df.copy()
    for k, v in defaults.items():
        if k not in out.columns:
            out[k] = v
    return out

anomaly_out = ensure_columns(
    anomaly_candidates.rename(columns={'anomaly_score': 'anomaly_score'}),
    {'peer_group_description': 'not_applicable'}
)
anomaly_out['streak_length'] = np.nan

deter_out = ensure_columns(
    deterioration_candidates.rename(columns={'streak_deterioration': 'streak_length'}),
    {'peer_group_description': 'not_applicable'}
)
deter_out['deterioration_score'] = deter_out['severity_score']

peer_out = ensure_columns(
    peer_candidates.rename(columns={
        'peer_group_size': 'peer_group_size',
        'peer_confidence': 'peer_confidence',
    }),
    {'peer_group_description': 'primary: COUNTRY|ZONE_TYPE|ZONE_PRIORITIZATION'}
)

opp_out = ensure_columns(
    opportunity_candidates.rename(columns={
        'peer_group_size': 'peer_group_size',
        'peer_confidence': 'peer_confidence',
        'entity_value': 'raw_value',
    }),
    {'peer_group_description': 'primary/fallback_behavior'}
)
opp_out['streak_length'] = opp_out['streak_improvement']

driver_out = ensure_columns(
    driver_candidates,
    {'peer_group_description': 'not_applicable'}
)
driver_out['association_strength'] = driver_out['abs_rho']
driver_out['metric_display_name'] = driver_out['metric_display_name'].fillna('Orders')

master = pd.concat([anomaly_out, deter_out, peer_out, opp_out, driver_out], ignore_index=True, sort=False)

score_weights = {
    'severity': 0.45,
    'confidence': 0.35,
    'business_priority': 0.20,
}

master['final_rank_score'] = (
    score_weights['severity'] * master['severity_score'].fillna(0)
    + score_weights['confidence'] * master['confidence_score'].fillna(0)
    + score_weights['business_priority'] * master['business_priority_score'].fillna(0)
 )

provisional_penalty = master['direction_confidence'].eq('provisional').map({True: 0.90, False: 1.0})
validation_penalty = master['validation_status'].eq('pending_business_validation').map({True: 0.90, False: 1.0})
low_peer_penalty = master['peer_confidence'].eq('low_confidence').map({True: 0.85, False: 1.0}).fillna(1.0)
master['final_rank_score'] = np.clip(master['final_rank_score'] * provisional_penalty * validation_penalty * low_peer_penalty, 0, 1)

master['insight_id'] = [f"INS-{i:06d}" for i in range(1, len(master) + 1)]

display(master[['insight_id', 'insight_category', 'detector_name', 'display_entity', 'metric_id', 'severity_score', 'confidence_score', 'business_priority_score', 'final_rank_score']].sort_values('final_rank_score', ascending=False).head(15))
print('master insights:', master.shape)

,insight_id,insight_category,detector_name,display_entity,metric_id,severity_score,confidence_score,business_priority_score,final_rank_score
56,INS-000057,anomaly,anomaly_point,PE | Arequipa | Cercado Arequipa,gross_profit_ue,1.0,0.686375,1.0,0.721087
52,INS-000053,anomaly,anomaly_point,PE | Lima | Campoy,gross_profit_ue,1.0,0.686375,1.0,0.721087
64,INS-000065,anomaly,anomaly_point,PE | Lima | Cono norte,gross_profit_ue,1.0,0.686375,1.0,0.721087
67,INS-000068,anomaly,anomaly_point,PE | Lima | San Juan de Lurigancho,gross_profit_ue,1.0,0.686375,1.0,0.721087
68,INS-000069,anomaly,anomaly_point,PE | Lima | Santa Anita,gross_profit_ue,1.0,0.686375,1.0,0.721087
8,INS-000009,anomaly,anomaly_point,MX | Monterrey | MTY_Apodaca_Huinalá,gross_profit_ue,1.0,0.686375,1.0,0.721087
35,INS-000036,anomaly,anomaly_point,PE | Lima | San Juan de Miraflores,gross_profit_ue,1.0,0.686375,1.0,0.721087
26,INS-000027,anomaly,anomaly_point,PE | Lima | Chorrillos,gross_profit_ue,1.0,0.686375,1.0,0.721087
25,INS-000026,anomaly,anomaly_point,PE | Lima | La Molina,gross_profit_ue,1.0,0.686375,1.0,0.721087
14,INS-000015,anomaly,anomaly_point,PE | Lima | Comas,gross_profit_ue,1.0,0.686375,1.0,0.721087


master insights: (12443, 85)


## 9) Normalizacion de outputs

Se unifican todos los detectores en una estructura maestra comun para consumo posterior por el chatbot (NB40).

Campos clave comunes:
- identificacion (`insight_id`, `insight_category`, `detector_name`)
- entidad (`entity_keys`, `display_entity`)
- metrica (`metric_id`, `metric_display_name`)
- scoring (`severity_score`, `confidence_score`, `business_priority_score`, `final_rank_score`)
- gobernanza (`validation_status`, `direction_confidence`, `caveats`)
- trazabilidad (`evidence_struct`, `peer_group_info`)

In [12]:
master_columns = [
    'insight_id', 'insight_category', 'detector_name',
    'entity_keys', 'display_entity',
    'metric_id', 'metric_display_name', 'time_context',
    'severity_score', 'confidence_score', 'business_priority_score', 'final_rank_score',
    'validation_status', 'direction_confidence',
    'evidence_struct', 'peer_group_info',
    'narrative_stub', 'recommendation_stub', 'caveats',
    'raw_value', 'wow_delta', 'anomaly_score', 'deterioration_score', 'streak_length',
    'peer_group_description', 'peer_group_size', 'peer_confidence', 'entity_value', 'peer_median', 'gap_abs', 'gap_pct',
    'target_metric', 'candidate_driver', 'association_strength', 'narrative_label',
]

for c in master_columns:
    if c not in master.columns:
        master[c] = np.nan if c.endswith('_score') else ''

master = master[master_columns].copy()
master = master.sort_values(['final_rank_score', 'severity_score'], ascending=[False, False]).reset_index(drop=True)

display(master.head(12))

,insight_id,insight_category,detector_name,entity_keys,display_entity,metric_id,metric_display_name,time_context,severity_score,confidence_score,business_priority_score,final_rank_score,validation_status,direction_confidence,evidence_struct,peer_group_info,narrative_stub,recommendation_stub,caveats,raw_value,wow_delta,anomaly_score,deterioration_score,streak_length,peer_group_description,peer_group_size,peer_confidence,entity_value,peer_median,gap_abs,gap_pct,target_metric,candidate_driver,association_strength,narrative_label
0,INS-000009,anomaly,anomaly_point,"{""COUNTRY"": ""MX"", ""CITY"": ""Monterrey"", ""ZONE"":...",MX | Monterrey | MTY_Apodaca_Huinalá,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.027261359300000065, ""median_value"": ...",,,,direction provisional; validation_status=pendi...,0.132192,-0.128137,24.921579,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
1,INS-000015,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""Com...",PE | Lima | Comas,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.16635739240000003, ""median_value"": 0...",,,,direction provisional; validation_status=pendi...,0.100919,-1.072339,-2.931346,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
2,INS-000026,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""La ...",PE | Lima | La Molina,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.06102350099999976, ""median_value"": 2...",,,,direction provisional; validation_status=pendi...,1.066340,-1.698861,-16.425413,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
3,INS-000027,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""Cho...",PE | Lima | Chorrillos,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.21619920299999995, ""median_value"": 1...",,,,direction provisional; validation_status=pendi...,0.548438,-1.206589,-2.708102,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
4,INS-000036,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""San...",PE | Lima | San Juan de Miraflores,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.1153168200000001, ""median_value"": 1....",,,,direction provisional; validation_status=pendi...,0.683788,-0.676518,-2.593471,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
5,INS-000053,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""Cam...",PE | Lima | Campoy,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.1374288001, ""median_value"": 0.345562...",,,,direction provisional; validation_status=pendi...,-0.798204,-1.475595,-5.613602,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
6,INS-000057,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Arequipa"", ""ZONE"": ...",PE | Arequipa | Cercado Arequipa,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.12333065999999993, ""median_value"": 1...",,,,direction provisional; validation_status=pendi...,0.608472,-0.861845,-3.244900,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
7,INS-000065,anomaly,anomaly_point,"{""COUNTRY"": ""PE"", ""CITY"": ""Lima"", ""ZONE"": ""Con...",PE | Lima | Cono norte,gross_profit_ue,Gross Profit UE,L0W,1.0,0.686375,1.0,0.721087,pending_business_validation,provisional,"{""mad"": 0.1587967105, ""median_value"": 0.875237...",,,,direction provisional; validation_status=pendi...,-0.029142,-1.213776,-3.841415,NaN,NaN,not_applicable,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,None
8,INS-000068,anom

## 10) Narrativa templada (sin LLM libre)

Cada insight se transforma en narrativa estructurada por plantilla y categoria:
- finding
- impacto potencial
- evidencia
- limite/caveat
- recomendacion preliminar

Regla: si la evidencia es debil o la direccion/proxy no esta confirmada, el texto lo explicita.

In [13]:
def make_narrative(row: pd.Series) -> tuple[str, str]:
    category = row['insight_category']
    metric = row['metric_display_name'] if pd.notna(row['metric_display_name']) else row['metric_id']
    caveat = row['caveats'] if pd.notna(row['caveats']) else ''

    if category == 'anomaly':
        finding = f"Se detecto una anomalia puntual en {metric} para {row['display_entity']}."
        impact = 'Puede indicar ruptura operacional reciente si se confirma con contexto.'
        evidence = f"modified_z={row['anomaly_score']:.2f}; wow_delta={row['wow_delta']:.4f}"
        rec = 'Validar cambios operativos/logisticos recientes en la zona y revisar si el cambio persiste la proxima semana.'
    elif category == 'persistent_deterioration':
        finding = f"Se detecto deterioro persistente en {metric} para {row['display_entity']}."
        impact = 'Riesgo de degradacion sostenida de performance si no se corrige.'
        evidence = f"streak_length={int(row['streak_length']) if pd.notna(row['streak_length']) else 'NA'}"
        rec = 'Priorizar analisis de causa operativa y definir accion correctiva por frente (oferta, conversion o calidad).'
    elif category == 'peer_gap':
        finding = f"La zona esta por debajo de su peer group en {metric}."
        impact = 'Puede indicar brecha competitiva local frente a zonas comparables.'
        evidence = f"gap_abs={row['gap_abs']:.4f}; gap_pct={row['gap_pct']:.4f}; peer_size={int(row['peer_group_size']) if pd.notna(row['peer_group_size']) else 'NA'}"
        rec = 'Comparar practicas operativas con peers lideres del mismo segmento y evaluar acciones replicables.'
    elif category == 'opportunity':
        finding = f"Se detecto oportunidad en {metric} para {row['display_entity']}."
        impact = 'Hay senal positiva potencialmente escalable si se confirma su estabilidad.'
        evidence = f"severity={row['severity_score']:.3f}; confidence={row['confidence_score']:.3f}"
        rec = 'Evaluar replicabilidad en zonas similares y priorizar seguimiento en high priority zones.'
    else:
        finding = f"Se observo asociacion de {row.get('candidate_driver')} con orders en {row['display_entity']}."
        impact = 'Puede orientar exploracion de posibles drivers, sin inferencia causal.'
        evidence = f"association_strength={row.get('association_strength', np.nan):.3f}; label={row.get('narrative_label')}"
        rec = 'Usar esta asociacion para priorizar investigacion adicional; no tomar decision final solo con esta evidencia.'

    narrative = (
        f"Finding: {finding}\n"
        f"Impacto potencial: {impact}\n"
        f"Evidencia: {evidence}\n"
        f"Limite/Caveat: {caveat}\n"
        f"Recomendacion preliminar: {rec}"
    )
    return narrative, rec

master[['narrative_stub', 'recommendation_stub']] = master.apply(
    lambda r: pd.Series(make_narrative(r)),
    axis=1,
 )

display(master[['insight_id', 'insight_category', 'display_entity', 'metric_id', 'narrative_stub']].head(5))

,insight_id,insight_category,display_entity,metric_id,narrative_stub
0,INS-000009,anomaly,MX | Monterrey | MTY_Apodaca_Huinalá,gross_profit_ue,Finding: Se detecto una anomalia puntual en Gr...
1,INS-000015,anomaly,PE | Lima | Comas,gross_profit_ue,Finding: Se detecto una anomalia puntual en Gr...
2,INS-000026,anomaly,PE | Lima | La Molina,gross_profit_ue,Finding: Se detecto una anomalia puntual en Gr...
3,INS-000027,anomaly,PE | Lima | Chorrillos,gross_profit_ue,Finding: Se detecto una anomalia puntual en Gr...
4,INS-000036,anomaly,PE | Lima | San Juan de Miraflores,gross_profit_ue,Finding: Se detecto una anomalia puntual en Gr...


## 11) Exportes

Se generan artefactos para consumo operativo y auditoria.

- Tabla maestra de candidatos (`parquet` + `csv`).
- Reporte tecnico (`md` + `json`).
- Muestras narrativas (`md`).

In [14]:
def markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return '(sin filas)'
    cols = [str(c) for c in df.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in df.iterrows():
        vals = [str(row[c]).replace('\n', ' ') for c in df.columns]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

master.to_parquet(REPORTS_DIR / 'insight_candidates.parquet', index=False)
master.to_csv(REPORTS_DIR / 'insight_candidates.csv', index=False)

summary_json = {
    'generated_at': datetime.utcnow().isoformat(),
    'n_candidates': int(master.shape[0]),
    'by_category': master['insight_category'].value_counts().to_dict(),
    'by_detector': master['detector_name'].value_counts().to_dict(),
    'score_weights': score_weights,
    'penalties': {
        'direction_confidence_provisional': 0.90,
        'validation_pending_business': 0.90,
        'peer_low_confidence': 0.85,
    },
    'governance': {
        'suspended_metrics': sorted(list(suspended_metrics)),
        'peer_restricted_metrics': sorted(list(peer_restricted_metrics)),
        'peer_dims': peer_dims,
        'thresholds': {
            'wow_threshold_pct': wow_threshold_pct,
            'streak_min_weeks': streak_min_weeks,
            'robust_z_threshold': robust_z_threshold,
            'min_peer_size': min_peer_size,
            'min_peer_warn': min_peer_warn,
        },
    },
    'open_limitations': [
        'Thresholds provisionales pendientes de calibracion por metrica',
        'Direction confidence aun provisional en el catalogo',
        '9 semanas limitan robustez temporal y de asociaciones',
        'Peer groups low_confidence requieren interpretacion cautelosa',
    ],
}

with open(REPORTS_DIR / 'insight_engine_report.json', 'w', encoding='utf-8') as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

top_ranked = master.head(20)[['insight_id', 'insight_category', 'detector_name', 'display_entity', 'metric_id', 'severity_score', 'confidence_score', 'final_rank_score', 'caveats']]
report_md = []
report_md.append('# Insight Engine Report - Reto 1')
report_md.append('')
report_md.append(f"Generado: {summary_json['generated_at']}")
report_md.append('')
report_md.append('## Resumen')
report_md.append(f"- n_candidates: {summary_json['n_candidates']}")
report_md.append(f"- by_category: {summary_json['by_category']}")
report_md.append(f"- by_detector: {summary_json['by_detector']}")
report_md.append('')
report_md.append('## Top candidatos')
report_md.append(markdown_table(top_ranked))
report_md.append('')
report_md.append('## Limites abiertos')
for lim in summary_json['open_limitations']:
    report_md.append(f"- {lim}")

with open(REPORTS_DIR / 'insight_engine_report.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_md))

samples = master.groupby('insight_category', dropna=False).head(3)[['insight_id', 'insight_category', 'display_entity', 'metric_id', 'narrative_stub', 'caveats']]
sample_md = ['# Insight Samples - Reto 1', '', markdown_table(samples)]
with open(REPORTS_DIR / 'insight_samples.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(sample_md))

print('Archivos generados en', REPORTS_DIR)
print([
    'insight_candidates.parquet',
    'insight_candidates.csv',
    'insight_engine_report.md',
    'insight_engine_report.json',
    'insight_samples.md',
])

Archivos generados en /home/thechieft/projects/IAeng/reports/reto1
['insight_candidates.parquet', 'insight_candidates.csv', 'insight_engine_report.md', 'insight_engine_report.json', 'insight_samples.md']


## 12) Validacion interna del engine

Objetivo: probar que los detectores disparan cuando deben y no disparan cuando no deben, incluyendo edge cases de gobernanza.

Casos auditados:
- metricas suspendidas
- peer groups too_small
- metrica lower_is_better
- direction provisional + caveat

In [15]:
positive_examples = master.head(10)[['insight_id', 'insight_category', 'detector_name', 'display_entity', 'metric_id', 'final_rank_score']]

flagged_keys = set(
    master.loc[master['metric_id'].ne('orders_total'), ['display_entity', 'metric_id']].itertuples(index=False, name=None)
 )
non_trigger_pool = latest[['COUNTRY', 'CITY', 'ZONE', 'id', 'display_name', 'raw_value', 'wow_delta']].copy()
non_trigger_pool['display_entity'] = non_trigger_pool.apply(build_display_entity, axis=1)
non_trigger_pool = non_trigger_pool.loc[
    ~non_trigger_pool.apply(lambda r: (r['display_entity'], r['id']) in flagged_keys, axis=1)
 ]
negative_examples = non_trigger_pool[['display_entity', 'id', 'display_name', 'raw_value', 'wow_delta']].head(10)

lower_metric = 'restaurants_markdowns_gmv'
markdown_peer = peer_out.loc[peer_out['metric_id'].eq(lower_metric)].copy()
lower_is_better_respected = True
if not markdown_peer.empty:
    # Para lower_is_better, underperformance debe corresponder mayor valor que el peer (gap_abs > 0).
    lower_is_better_respected = bool((markdown_peer['gap_abs'] > 0).all())

validation_checks = pd.DataFrame([
    {'check': 'semantic_checks_fail_count_zero', 'result': n_fail == 0, 'detail': f'FAIL={n_fail}, WARN={n_warn}'},
    {'check': 'suspended_metrics_not_emitted', 'result': master['metric_id'].isin(suspended_metrics).sum() == 0, 'detail': f"count={int(master['metric_id'].isin(suspended_metrics).sum())}"},
    {'check': 'turbo_adoption_not_in_peer_gap', 'result': peer_out['metric_id'].eq('turbo_adoption').sum() == 0, 'detail': f"count={int(peer_out['metric_id'].eq('turbo_adoption').sum())}"},
    {'check': 'peer_gap_excludes_too_small', 'result': peer_out['peer_group_size'].fillna(999).lt(min_peer_warn).sum() == 0, 'detail': f"count={int(peer_out['peer_group_size'].fillna(999).lt(min_peer_warn).sum())}"},
    {'check': 'lower_is_better_rule_respected', 'result': lower_is_better_respected, 'detail': 'restaurants_markdowns_gmv evaluada contra peer gap'},
    {'check': 'provisional_direction_has_caveat', 'result': master.loc[master['direction_confidence'].eq('provisional'), 'caveats'].str.contains('direction provisional', na=False).all(), 'detail': 'se verifica caveat de direccion provisional'},
])

display(validation_checks)
print('--- Positivos (disparan) ---')
display(positive_examples)
print('--- Negativos (no disparan) ---')
display(negative_examples)

,check,result,detail
0,semantic_checks_fail_count_zero,True,"FAIL=0, WARN=0"
1,suspended_metrics_not_emitted,True,count=0
2,turbo_adoption_not_in_peer_gap,True,count=0
3,peer_gap_excludes_too_small,True,count=0
4,lower_is_better_rule_respected,True,restaurants_markdowns_gmv evaluada contra peer...
5,provisional_direction_has_caveat,True,se verifica caveat de direccion provisional


--- Positivos (disparan) ---


,insight_id,insight_category,detector_name,display_entity,metric_id,final_rank_score
0,INS-000009,anomaly,anomaly_point,MX | Monterrey | MTY_Apodaca_Huinalá,gross_profit_ue,0.721087
1,INS-000015,anomaly,anomaly_point,PE | Lima | Comas,gross_profit_ue,0.721087
2,INS-000026,anomaly,anomaly_point,PE | Lima | La Molina,gross_profit_ue,0.721087
3,INS-000027,anomaly,anomaly_point,PE | Lima | Chorrillos,gross_profit_ue,0.721087
4,INS-000036,anomaly,anomaly_point,PE | Lima | San Juan de Miraflores,gross_profit_ue,0.721087
5,INS-000053,anomaly,anomaly_point,PE | Lima | Campoy,gross_profit_ue,0.721087
6,INS-000057,anomaly,anomaly_point,PE | Arequipa | Cercado Arequipa,gross_profit_ue,0.721087
7,INS-000065,anomaly,anomaly_point,PE | Lima | Cono norte,gross_profit_ue,0.721087
8,INS-000068,anomaly,anomaly_point,PE | Lima | San Juan de Lurigancho,gross_profit_ue,0.721087
9,INS-000069,anomaly,anomaly_point,PE | Lima | Santa Anita,gross_profit_ue,0.721087


--- Negativos (no disparan) ---


,display_entity,id,display_name,raw_value,wow_delta
3,CL | Santiago De Chile | Colina,retail_sst_ss_cvr,Retail SST > SS CVR,0.741457,0.007542
5,MX | Puebla | Angelopolis,gross_profit_ue,Gross Profit UE,1.329837,-0.181975
6,BR | Juiz De Fora | MG - JDF - Juiz de Fora,retail_sst_ss_cvr,Retail SST > SS CVR,0.898542,-0.021600
15,MX | Piedras Negras | PDN_Piedras_Negras,retail_sst_ss_cvr,Retail SST > SS CVR,0.856905,-0.002901
25,AR | Buenos Aires | Lomas de Zamora - Banfield,retail_sst_ss_cvr,Retail SST > SS CVR,0.909954,-0.004413
26,CL | Rancagua | Rancagua Centro,gross_profit_ue,Gross Profit UE,-4.582923,-0.529094
29,CL | Santiago De Chile | Población San Joaquin,retail_sst_ss_cvr,Retail SST > SS CVR,0.760478,-0.025558
30,CO | Barranquilla | Barranquilla sur,gross_profit_ue,Gross Profit UE,2.159952,-0.236004
36,MX | Cuautla | Cuautla,retail_sst_ss_cvr,Retail SST > SS CVR,0.887300,-0.004982
42,MX | Campeche | CAMP Centro,retail_sst_ss_cvr,Retail SST > SS CVR,0.906807,-0.009805


## 13) Proximos pasos

### Que quedo listo para NB40
- Contrato de salida normalizado para consumo de chatbot.
- Detectores trazables con scoring y caveats.
- Narrativa templada sin depender de LLM libre.

### Que sigue abierto
- Calibracion de thresholds por metrica.
- Validacion de desired_direction con negocio.
- Politica final para uso de peer fallback adicional.

### Que debe consumir NB40
- `insight_candidates.*` como backend de hallazgos.
- `insight_engine_report.*` como metadata de gobernanza.
- `insight_samples.md` como ejemplos de narrativa controlada.